# 03 — Exploratory Data Analysis (EDA)

Comprehensive visualizations of stock and mutual fund data:

1. Stock price trends (line chart)
2. Normalized price comparison
3. Daily return distribution (histogram)
4. Volatility comparison (bar chart)
5. Stock risk vs return (scatter)
6. Mutual fund category distribution
7. Top mutual funds by returns
8. Mutual fund risk vs return (scatter)

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os, sys
import warnings
warnings.filterwarnings('ignore')

# Style configuration
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12

# Add src/ to path
sys.path.insert(0, os.path.join('..', 'src'))

from data_processing import load_and_clean_all_stocks, load_raw_mutual_funds, clean_mutual_funds
from metrics import compute_stock_metrics, compute_mf_metrics

RAW_DIR = os.path.join('..', 'data', 'raw')
TICKERS = ['AAPL', 'MSFT', 'GOOG']

In [ ]:
# Load cleaned data
stocks = load_and_clean_all_stocks(TICKERS, RAW_DIR)
mf = clean_mutual_funds(load_raw_mutual_funds(RAW_DIR))

print(f'Stock data : {stocks.shape[0]} rows, {stocks["ticker"].nunique()} tickers')
print(f'Mutual fund: {mf.shape[0]} schemes, {mf["category"].nunique()} categories')

---
## 1. Stock Price Trends

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

colors = {'AAPL': '#1f77b4', 'MSFT': '#ff7f0e', 'GOOG': '#2ca02c'}

for ticker in TICKERS:
 subset = stocks[stocks['ticker'] == ticker]
 ax.plot(subset['date'], subset['close'], label=ticker,
   color=colors[ticker], linewidth=1.5, alpha=0.9)

ax.set_title(' Stock Price Trends Over Time', fontsize=16, fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('Close Price ($)')
ax.legend(fontsize=12, loc='upper left')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../data/processed/plot_stock_price_trends.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. Normalized Price Comparison

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

for ticker in TICKERS:
 subset = stocks[stocks['ticker'] == ticker].copy()
 # Normalize: (price / first_price) * 100
 first_price = subset['close'].iloc[0]
 subset['normalized'] = (subset['close'] / first_price) * 100
 ax.plot(subset['date'], subset['normalized'], label=ticker,
   color=colors[ticker], linewidth=1.5)

ax.axhline(y=100, color='gray', linestyle='--', alpha=0.5, label='Baseline (100)')
ax.set_title(' Normalized Price Comparison (Base = 100)', fontsize=16, fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('Normalized Price')
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../data/processed/plot_normalized_prices.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Daily Return Distribution

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)

for i, ticker in enumerate(TICKERS):
 subset = stocks[stocks['ticker'] == ticker]['daily_return'].dropna()
 axes[i].hist(subset, bins=60, color=colors[ticker], alpha=0.7, edgecolor='white')
 axes[i].axvline(x=0, color='red', linestyle='--', alpha=0.6)
 axes[i].axvline(x=subset.mean(), color='black', linestyle='-', alpha=0.8,
     label=f'Mean: {subset.mean():.3f}%')
 axes[i].set_title(f'{ticker} Daily Returns', fontweight='bold')
 axes[i].set_xlabel('Daily Return (%)')
 axes[i].legend(fontsize=10)

axes[0].set_ylabel('Frequency')
fig.suptitle(' Daily Return Distributions', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../data/processed/plot_daily_return_dist.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Volatility Comparison

In [ ]:
# Compute metrics
stock_metrics = compute_stock_metrics(stocks)
display(stock_metrics)

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.bar(stock_metrics['ticker'], stock_metrics['volatility'],
    color=[colors[t] for t in stock_metrics['ticker']],
    edgecolor='white', linewidth=2, width=0.5)

# Add value labels
for bar, val in zip(bars, stock_metrics['volatility']):
 ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
   f'{val:.4f}%', ha='center', fontsize=12, fontweight='bold')

ax.set_title(' Volatility Comparison (Std Dev of Daily Returns)', fontsize=16, fontweight='bold')
ax.set_xlabel('Stock')
ax.set_ylabel('Volatility (σ %)')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('../data/processed/plot_volatility_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Stock Risk vs Return (Scatter Plot)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))

for _, row in stock_metrics.iterrows():
 ax.scatter(row['volatility'], row['avg_daily_return'],
    s=300, color=colors[row['ticker']], edgecolors='black',
    linewidth=1.5, zorder=5)
 ax.annotate(row['ticker'],
    (row['volatility'], row['avg_daily_return']),
    textcoords='offset points', xytext=(12, 8),
    fontsize=14, fontweight='bold')

ax.set_title(' Risk vs Return — Stocks', fontsize=16, fontweight='bold')
ax.set_xlabel('Volatility (σ %)')
ax.set_ylabel('Average Daily Return (%)')
ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
ax.grid(True, alpha=0.3)

# Add quadrant labels
ax.text(0.02, 0.98, 'Low Risk\nHigh Return ★', transform=ax.transAxes,
  fontsize=9, alpha=0.4, va='top', ha='left', style='italic')
ax.text(0.98, 0.02, 'High Risk\nLow Return', transform=ax.transAxes,
  fontsize=9, alpha=0.4, va='bottom', ha='right', style='italic')

plt.tight_layout()
plt.savefig('../data/processed/plot_stock_risk_return.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 6. Mutual Fund Category Distribution

In [ ]:
cat_counts = mf['category'].value_counts().head(15)

fig, ax = plt.subplots(figsize=(14, 7))
palette = sns.color_palette('viridis', len(cat_counts))
bars = ax.barh(cat_counts.index[::-1], cat_counts.values[::-1],
    color=palette[::-1], edgecolor='white', linewidth=1)

# Value labels
for bar, val in zip(bars, cat_counts.values[::-1]):
 ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2,
   str(val), va='center', fontsize=11, fontweight='bold')

ax.set_title(' Mutual Fund Category Distribution', fontsize=16, fontweight='bold')
ax.set_xlabel('Number of Funds')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('../data/processed/plot_mf_category_dist.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Top Mutual Funds by 1-Year Return

In [ ]:
top_n = 15
top_mf = mf.nlargest(top_n, 'returns_1yr')

fig, ax = plt.subplots(figsize=(14, 8))

# Truncate long names
labels = [name[:40] + '...' if len(name) > 40 else name
   for name in top_mf['scheme_name']]

palette = sns.color_palette('YlOrRd_r', top_n)
bars = ax.barh(labels[::-1], top_mf['returns_1yr'].values[::-1],
    color=palette[::-1], edgecolor='white', linewidth=1)

for bar, val in zip(bars, top_mf['returns_1yr'].values[::-1]):
 ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
   f'{val:.1f}%', va='center', fontsize=10, fontweight='bold')

ax.set_title(f' Top {top_n} Mutual Funds by 1-Year Return', fontsize=16, fontweight='bold')
ax.set_xlabel('1-Year Return (%)')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('../data/processed/plot_top_mf_returns.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Mutual Fund Risk vs Return (Scatter)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 8))

# Color by risk level
risk_colors = {
 'Low': '#2ecc71', 'Moderately Low': '#27ae60',
 'Moderate': '#f39c12', 'Moderately High': '#e67e22',
 'High': '#e74c3c', 'Very High': '#c0392b'
}

for risk_level in mf['risk_level'].unique():
 subset = mf[mf['risk_level'] == risk_level]
 color = risk_colors.get(risk_level, '#95a5a6')
 ax.scatter(subset['sd'], subset['returns_1yr'],
    alpha=0.6, s=60, label=risk_level,
    color=color, edgecolors='white', linewidth=0.5)

# Add trend line
valid = mf[['sd', 'returns_1yr']].dropna()
z = np.polyfit(valid['sd'], valid['returns_1yr'], 1)
p = np.poly1d(z)
x_line = np.linspace(valid['sd'].min(), valid['sd'].max(), 100)
ax.plot(x_line, p(x_line), '--', color='red', alpha=0.7,
  linewidth=2, label='Trend Line')

corr = valid['sd'].corr(valid['returns_1yr'])
ax.set_title(f' Mutual Fund Risk vs Return (r = {corr:.2f})',
    fontsize=16, fontweight='bold')
ax.set_xlabel('Risk (Standard Deviation)')
ax.set_ylabel('1-Year Return (%)')
ax.legend(title='Risk Level', fontsize=10, title_fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../data/processed/plot_mf_risk_return.png', dpi=150, bbox_inches='tight')
plt.show()

## Bonus: Correlation Heatmap

In [ ]:
# Mutual Fund numeric correlation
numeric_cols = ['returns_1yr', 'returns_3yr', 'returns_5yr', 'sd',
    'sharpe', 'sortino', 'alpha', 'beta', 'expense_ratio']
available_cols = [c for c in numeric_cols if c in mf.columns]

fig, ax = plt.subplots(figsize=(10, 8))
corr_matrix = mf[available_cols].corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f',
   cmap='RdYlGn', center=0, ax=ax,
   square=True, linewidths=1)
ax.set_title(' Mutual Fund Metrics — Correlation Heatmap',
    fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('../data/processed/plot_mf_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

## Summary

### Key Findings:
- Stock prices show distinct long-term trends with varying growth trajectories
- Daily returns follow approximately normal distributions with slight skewness
- Risk-return relationship is evident across both stocks and mutual funds
- Mutual fund categories show significant variation in both risk and return profiles

### Next Steps
→ Proceed to **04_llm_insights.ipynb** for AI-generated financial insights